# goit-de-hw-03 / goit-de-hw-04
Запуск у Google Colab (PySpark 3.5.3).

In [ ]:
!pip -q install pyspark==3.5.3 gdown
!gdown -q 1GN8o8vYP8LLj3A0UWkvbgnf0YjcF3PkV -O users.csv
!gdown -q 1im5cGVl1z8ejypTyNJZSQ2lqMGh6tTFN -O products.csv
!gdown -q 1cDLTc0KYQfnsDWHLR0cJkF5puBpAo2Uh -O purchases.csv
!gdown -q 1XmmUpuTwNH6YZHZMVRYQe0vJvBy3vQY5 -O nuek-vuh3.csv
!ls -la *.csv; java -version

## ДЗ 3

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round, sum as spark_sum

# Створюємо сесію Spark
spark = SparkSession.builder.appName("goit-de-hw-03").getOrCreate()

In [ ]:
# 1. Завантажуємо та читаємо кожен CSV-файл як окремий DataFrame
users_df = spark.read.csv("./users.csv", header=True, inferSchema=True)
purchases_df = spark.read.csv("./purchases.csv", header=True, inferSchema=True)
products_df = spark.read.csv("./products.csv", header=True, inferSchema=True)

print("1. Початкові дані")
users_df.show(5)
purchases_df.show(5)
products_df.show(5)

In [ ]:
# 2. Очищуємо дані: видаляємо рядки з пропущеними значеннями
users_clean = users_df.dropna()
purchases_clean = purchases_df.dropna()
products_clean = products_df.dropna()

print("2. Кількість рядків до / після очищення")
print("users:", users_df.count(), "->", users_clean.count())
print("purchases:", purchases_df.count(), "->", purchases_clean.count())
print("products:", products_df.count(), "->", products_clean.count())
users_clean.show(5)
purchases_clean.show(5)
products_clean.show(5)

In [ ]:
# Об'єднуємо покупки з продуктами і рахуємо суму кожної покупки
purchases_products = purchases_clean \
    .join(products_clean, on="product_id", how="inner") \
    .withColumn("total_price", col("quantity") * col("price"))

In [ ]:
# 3. Загальна сума покупок за кожною категорією продуктів
total_by_category = purchases_products \
    .groupBy("category") \
    .agg(spark_round(spark_sum("total_price"), 2).alias("total_purchases")) \
    .orderBy(col("total_purchases").desc())

print("3. Загальна сума покупок за категоріями")
total_by_category.show()

In [ ]:
# Додаємо дані користувачів і залишаємо лише вік 18-25 включно
purchases_18_25 = purchases_products \
    .join(users_clean, on="user_id", how="inner") \
    .where((col("age") >= 18) & (col("age") <= 25))

In [ ]:
# 4. Сума покупок за кожною категорією для віку 18-25
total_by_category_18_25 = purchases_18_25 \
    .groupBy("category") \
    .agg(spark_round(spark_sum("total_price"), 2).alias("total_purchases_18_25")) \
    .orderBy(col("total_purchases_18_25").desc())

print("4. Сума покупок за категоріями (вік 18-25)")
total_by_category_18_25.show()

In [ ]:
# 5. Частка покупок кожної категорії від сумарних витрат (вік 18-25)
total_spent_18_25 = purchases_18_25.agg(spark_sum("total_price")).collect()[0][0]

share_by_category_18_25 = total_by_category_18_25 \
    .withColumn(
        "percentage",
        spark_round(col("total_purchases_18_25") / total_spent_18_25 * 100, 2),
    ) \
    .orderBy(col("percentage").desc())

print("5. Частка покупок за категоріями (вік 18-25), %")
print("Сумарні витрати 18-25:", round(total_spent_18_25, 2))
share_by_category_18_25.show()

In [ ]:
# 6. Топ-3 категорії з найвищим відсотком витрат (вік 18-25)
top_3_categories = share_by_category_18_25 \
    .select("category", "percentage") \
    .orderBy(col("percentage").desc()) \
    .limit(3)

print("6. Топ-3 категорії (вік 18-25)")
top_3_categories.show()

In [ ]:
# Закриваємо сесію Spark
spark.stop()

## ДЗ 4

### Частина 1

In [ ]:
from pyspark.sql import SparkSession

# Створюємо сесію Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .appName("MyGoitSparkSandbox") \
    .getOrCreate()

# Завантажуємо датасет
nuek_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('./nuek-vuh3.csv')

nuek_repart = nuek_df.repartition(2)

nuek_processed = nuek_repart \
    .where("final_priority < 3") \
    .select("unit_id", "final_priority") \
    .groupBy("unit_id") \
    .count()

# Ось ТУТ додано рядок
nuek_processed = nuek_processed.where("count>2")

nuek_processed.collect()


print("Spark", spark.version)

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(4040, path='/jobs/')

In [ ]:
spark.stop()

### Частина 2

In [ ]:
from pyspark.sql import SparkSession

# Створюємо сесію Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .appName("MyGoitSparkSandbox") \
    .getOrCreate()

# Завантажуємо датасет
nuek_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('./nuek-vuh3.csv')

nuek_repart = nuek_df.repartition(2)

nuek_processed = nuek_repart \
    .where("final_priority < 3") \
    .select("unit_id", "final_priority") \
    .groupBy("unit_id") \
    .count()

# Проміжний action: collect
nuek_processed.collect()

# Ось ТУТ додано рядок
nuek_processed = nuek_processed.where("count>2")

nuek_processed.collect()


print("Spark", spark.version)

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(4040, path='/jobs/')

In [ ]:
spark.stop()

### Частина 3

In [ ]:
from pyspark.sql import SparkSession

# Створюємо сесію Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .appName("MyGoitSparkSandbox") \
    .getOrCreate()

# Завантажуємо датасет
nuek_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('./nuek-vuh3.csv')

nuek_repart = nuek_df.repartition(2)

nuek_processed_cached = nuek_repart \
    .where("final_priority < 3") \
    .select("unit_id", "final_priority") \
    .groupBy("unit_id") \
    .count() \
    .cache()  # Додано функцію cache

# Проміжний action: collect
nuek_processed_cached.collect()

# Ось ТУТ додано рядок
nuek_processed = nuek_processed_cached.where("count>2")

nuek_processed.collect()



print("Spark", spark.version)

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(4040, path='/jobs/')

In [ ]:
nuek_processed_cached.unpersist()
spark.stop()